# The quantum numbers in qg terms, and Λ as a filter on LiH (§155, §153)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_quantum_numbers.ipynb)

**Part A (§155)** reads each quantum number of an electron with qang: m_s from one qubit's qg_Z, m_l from the Hamming weight of a symmetric register (so the qg weight filter is the projection onto one m_l), l from the qang value of SWAP, and the hydrogen shell n as two spins (n−1)/2. Exact identities, shown on sampled circuits.

**Part B (§153)** uses the molecular m_l, Λ, as a third filter on a 10-qubit LiH ground state, next to the electron-number filters, under the FakeTorino noise model.

Everything runs on local simulators (no account, no hardware), in about 15 minutes (most of it in the LiH part, Part B).

## 1. Install

In [1]:
import importlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qang", "qiskit-aer", "qiskit-ibm-runtime", "scipy"], check=False)
importlib.invalidate_caches()
try:
    from qang import angular, ibm
    ok = hasattr(angular, "j_squared_from_swaps") and hasattr(ibm, "best_chain")
except ImportError:
    ok = False
if not ok:  # the published qang predates §155: install from GitHub
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                    "git+https://github.com/Viny2030/qang.git"], check=True)
    print("installed qang from GitHub; if an import fails below, restart the runtime once")
import qang
print("qang", qang.__version__)

qang 0.10.0


## 2. Shared set-up

`orbital_circuit` prepares the l = 1 orbitals and the l = 0 singlet on two qubits (m_l = 1 − Hamming weight); `with_measure(..., bell=True)` rotates the Bell basis so that the singlet is the outcome 11, which reads qg_SWAP = ⟨SWAP⟩ = 1 − 2 P(singlet). Copied from `examples/quantum_numbers_qg.py`.

In [2]:
import math, time
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qang import angular as A
from qang import interop as IO
from qang import ibm
from qang.core import Qang

SHOTS = 20000
T0 = time.time()


def orbital_circuit(name):
    qc = QuantumCircuit(2)
    if name == "p-1":
        qc.x([0, 1])
    elif name in ("p0", "pz"):  # (|01> + |10>)/sqrt2
        qc.h(0); qc.cx(0, 1); qc.x(1)
    elif name == "px":  # proportional to |00> - |11>
        qc.h(0); qc.z(0); qc.cx(0, 1)
    elif name == "py":  # proportional to |00> + |11>
        qc.h(0); qc.cx(0, 1)
    elif name == "s":  # singlet (|01> - |10>)/sqrt2, l = 0
        qc.h(0); qc.z(0); qc.cx(0, 1); qc.x(1)
    return qc


def with_measure(qc, bell=False):
    c = qc.copy()
    if bell:
        c.cx(0, 1)
        c.h(0)
    c.measure_all()
    return c


def sample(circs, backend=None, seed=7):
    sim = AerSimulator.from_backend(backend, seed_simulator=seed) if backend else AerSimulator(seed_simulator=seed)
    t = transpile(circs, sim, optimization_level=1, seed_transpiler=137)
    res = sim.run(t, shots=SHOTS).result()
    return [IO.canonical_counts(res.get_counts(i), sdk="qiskit") for i in range(len(circs))]


def m_distribution(counts, n):
    pm = np.zeros(n + 1)
    shots = sum(counts.values())
    for k, c in counts.items():
        pm[sum(k)] += c / shots
    return pm


def qg_swap(bell_counts):
    shots = sum(bell_counts.values())
    return 1 - 2 * bell_counts.get((1, 1), 0) / shots

## 3. m_s: one qubit (§155)

For a spin ½, qg_Z = 2 m_s and the qang angle is the tilt of the spin. Published: RY(60°) gives qg_Z = +0.502 and the angle 59.9°.

In [3]:
for theta in (0.0, math.pi / 3, math.pi / 2, math.pi):
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    (counts,) = sample([with_measure(qc)])
    z, se = IO.qg_z_profile(counts)[0]
    q = Qang(z)
    print(f"RY({math.degrees(theta):5.1f} deg): qg_Z {q.value:+.3f} = {q.milliqang:+7.1f} mqg, "
          f"qang angle {q.to_theta('deg'):5.1f} deg, <m_s> = {z / 2:+.3f}, qg_S = {q.to_entropic().value:.3f}")

RY(  0.0 deg): qg_Z +1.000 = +1000.0 mqg, qang angle   0.0 deg, <m_s> = +0.500, qg_S = 0.000
RY( 60.0 deg): qg_Z +0.502 =  +501.8 mqg, qang angle  59.9 deg, <m_s> = +0.251, qg_S = 0.810


RY( 90.0 deg): qg_Z +0.002 =    +2.0 mqg, qang angle  89.9 deg, <m_s> = +0.001, qg_S = 1.000


RY(180.0 deg): qg_Z -1.000 = -1000.0 mqg, qang angle 180.0 deg, <m_s> = -0.500, qg_S = 0.000


## 4. m_l and l: a two-qubit register (§155)

An l = 1 orbital is a spin 1 = two qubits in their symmetric subspace: m_l = 1 − Hamming weight, and the register qg_Z (mean over its qubits) is ⟨m_l⟩/l. The real p_x and p_y are superpositions of m_l = ±1: their qg_Z is 0 like p_z, but their m_l distribution is spread (qg_S(m) = log 2 / log 3 = 0.63), which is why §153 needs the complex π± orbitals. l comes from SWAP: ⟨J²⟩ = 1 + qg_SWAP, so qg_SWAP = +1 means l = 1 and −1 means l = 0 (the singlet).

In [4]:
names = ("p+1", "p0", "p-1", "px", "py", "pz", "s")
zc = sample([with_measure(orbital_circuit(n)) for n in names])
bc = sample([with_measure(orbital_circuit(n), bell=True) for n in names])
print("orbital  P(m_l=+1,0,-1)        register qg_Z  qg_S(m)  qg_SWAP  l")
for name, cz, cb in zip(names, zc, bc):
    pm = m_distribution(cz, 2)  # index = weight: m_l = +1, 0, -1
    reg = float(np.mean([z for z, _ in IO.qg_z_profile(cz)]))
    sw = qg_swap(cb)
    l_hat = A.l_from_j_squared(A.j_squared_from_swaps(2, [sw]))
    print(f"{name:7s}  {np.round(pm, 3)!s:22s} {reg:+.3f}         {A.qg_s_spin(pm):.3f}    {sw:+.3f}   {l_hat:.2f}")

orbital  P(m_l=+1,0,-1)        register qg_Z  qg_S(m)  qg_SWAP  l
p+1      [1. 0. 0.]             +1.000         -0.000    +1.000   1.00
p0       [0. 1. 0.]             +0.000         -0.000    +1.000   1.00
p-1      [0. 0. 1.]             -1.000         -0.000    +1.000   1.00
px       [0.499 0.    0.501]    -0.001         0.631    +1.000   1.00
py       [0.499 0.    0.501]    -0.002         0.631    +1.000   1.00
pz       [0. 1. 0.]             +0.000         -0.000    +1.000   1.00
s        [0. 1. 0.]             -0.000         -0.000    -1.000   0.00


## 5. The shell n (§155)

Hydrogen's hidden SO(4) symmetry makes shell n two spins (n−1)/2: two symmetric registers of n−1 qubits each. For n = 2 that is the two-qubit register above: (qg_SWAP, register qg_Z) gives 2s and the three 2p. For n = 3 (exact states on 4 qubits) the register qg_Z is m_l/2, so the qang angles are 0°, 60°, 90°, 120°, 180°.

The qang angle of an eigenstate |l, m⟩ is arccos(m/l); the textbook vector-model angle arccos(m/√(l(l+1))) always lies closer to the equator, because the state has transverse spread l(l+1) − m².

In [5]:
labels = {"2s": "s", "2p+1": "p+1", "2p0": "p0", "2p-1": "p-1"}
zc = sample([with_measure(orbital_circuit(n)) for n in labels.values()])
bc = sample([with_measure(orbital_circuit(n), bell=True) for n in labels.values()])
print("n = 2:")
for label, cz, cb in zip(labels, zc, bc):
    sw = qg_swap(cb)
    reg = float(np.mean([z for z, _ in IO.qg_z_profile(cz)]))
    l_hat = round(A.l_from_j_squared(A.j_squared_from_swaps(2, [sw])))
    print(f"  {label:5s} qg_SWAP {sw:+.3f}, register qg_Z {reg:+.3f}  ->  l = {l_hat}, m_l = {round(reg):+d}")

print("n = 3 (two symmetric 2-qubit registers, 9 states):")
k = 2
sym = np.array([A.dicke_state(k, w) for w in range(k + 1)]).T
P = np.kron(sym, sym)
Jx, Jy, Jz = A.spin_operators(2 * k)
J2op = Jx @ Jx + Jy @ Jy + Jz @ Jz
ev, vec = np.linalg.eigh(P.conj().T @ J2op @ P + 1e-3 * P.conj().T @ Jz @ P)  # split the m inside each l
rows = []
for v in vec.T:
    psi = P @ v
    l = (-1 + math.sqrt(1 + 4 * np.real(np.vdot(psi, J2op @ psi)))) / 2
    z = np.real(np.vdot(psi, Jz @ psi)) / k
    rows.append((round(l), round(z * k), z))
for l, m, z in sorted(rows, key=lambda r: (r[0], -r[1])):
    print(f"  l = {l}, m_l = {m:+d}: register qg_Z {z:+.3f}, qang angle {Qang(z).to_theta('deg'):5.1f} deg")

print("qang angle vs vector-model angle:")
for l, m in ((1, 1), (2, 2), (2, 1), (3, 3), (3, 1)):
    print(f"  l = {l}, m = {m:+d}: qang {math.degrees(A.qang_angle_of_eigenstate(l, m)):5.1f} deg, "
          f"vector model {math.degrees(A.vector_model_angle(l, m)):5.1f} deg, <Jx^2+Jy^2> = {A.transverse_spread(l, m):.0f}")

n = 2:
  2s    qg_SWAP -1.000, register qg_Z +0.000  ->  l = 0, m_l = +0
  2p+1  qg_SWAP +1.000, register qg_Z +1.000  ->  l = 1, m_l = +1
  2p0   qg_SWAP +1.000, register qg_Z +0.000  ->  l = 1, m_l = +0
  2p-1  qg_SWAP +1.000, register qg_Z -1.000  ->  l = 1, m_l = -1
n = 3 (two symmetric 2-qubit registers, 9 states):
  l = 0, m_l = +0: register qg_Z -0.000, qang angle  90.0 deg
  l = 1, m_l = +1: register qg_Z +0.500, qang angle  60.0 deg
  l = 1, m_l = +0: register qg_Z -0.000, qang angle  90.0 deg
  l = 1, m_l = -1: register qg_Z -0.500, qang angle 120.0 deg
  l = 2, m_l = +2: register qg_Z +1.000, qang angle   0.0 deg
  l = 2, m_l = +1: register qg_Z +0.500, qang angle  60.0 deg
  l = 2, m_l = +0: register qg_Z -0.000, qang angle  90.0 deg
  l = 2, m_l = -1: register qg_Z -0.500, qang angle 120.0 deg
  l = 2, m_l = -2: register qg_Z -1.000, qang angle 180.0 deg
qang angle vs vector-model angle:
  l = 1, m = +1: qang   0.0 deg, vector model  45.0 deg, <Jx^2+Jy^2> = 1
  l = 2, m = 

## 6. With noise: the weight filter is the m_l = 0 projection (§155)

p_z has m_l = 0 exactly. Under the FakeTorino noise model it leaks to m_l = ±1, in 19% of the shots on the default layout (whose qubit 0 has a 17% readout error) and in about 2% on the chain chosen by `ibm.best_chain`, which also predicts the kept fraction. In both cases the weight filter returns P(m_l = 0) = 1.

In [6]:
import qiskit_ibm_runtime.fake_provider as fp

b = fp.FakeTorino()
sim = AerSimulator.from_backend(b, seed_simulator=7)
qc = with_measure(orbital_circuit("pz"))
default = transpile(qc, b, optimization_level=1, seed_transpiler=137)
layout, pred, chosen = ibm.best_chain(qc, b, weight=1)
for label, t in (("default layout", default), ("ibm.best_chain", chosen)):
    phys = ibm.measured_physical_qubits(t)
    cz = IO.canonical_counts(sim.run(t, shots=SHOTS).result().get_counts(), sdk="qiskit")
    pm = m_distribution(cz, 2)
    kept, frac = IO.weight_filter(cz, 1)
    pmf = m_distribution(kept, 2)
    print(f"{label} {phys}: readout errors {[round(max(e), 3) for e in ibm.readout_errors(b, phys)]}")
    print(f"  raw      P(m_l=+1,0,-1) = {np.round(pm, 4)}")
    print(f"  filtered P(m_l=+1,0,-1) = {np.round(pmf, 4)}, kept {frac:.3f}"
          + (f" (predicted {pred['kept']:.3f})" if t is chosen else ""))

default layout [0, 1]: readout errors [0.168, 0.023]
  raw      P(m_l=+1,0,-1) = [0.0924 0.8128 0.0948]
  filtered P(m_l=+1,0,-1) = [0. 1. 0.], kept 0.813


ibm.best_chain [92, 99]: readout errors [0.009, 0.012]
  raw      P(m_l=+1,0,-1) = [0.0117 0.978  0.0103]
  filtered P(m_l=+1,0,-1) = [0. 1. 0.], kept 0.978 (predicted 0.977)


## 7. Λ as a third filter on LiH (§153)

A linear molecule conserves Λ = Σ m_l(i) n_i. With the π orbitals written as π± (m_l = ±1) instead of p_x, p_y, Λ is diagonal and can be checked shot by shot, like the electron numbers N↑, N↓, and it only needs the four π qubits in the Z basis, so it also reaches off-diagonal measurement groups. LiH, 1.6 Å, STO-3G, frozen 1s, 10 qubits; the circuit is a unary loader over the five electron pairs (published θ, 5.0 mHa above CASCI). The Hamiltonian (PySCF, `tools/make_lih_lambda_hamiltonian.py`) is read from the repository.

Published on FakeTorino (20 000 shots per group, 317 groups): energy error raw 163.8 mHa, N filter 11.9, Λ alone 147.4, both 11.7; π occupation raw 0.138, both 0.0023 (ideal 0.002). Where N already applies Λ adds almost nothing (an error that keeps N↑, N↓ but changes Λ takes two errors); it is the right check on the π occupation. Here: 4 000 shots per group.

In [7]:
import json, os, urllib.request
from qiskit.circuit.library import XXPlusYYGate
from qiskit.quantum_info import SparsePauliOp, Statevector


def repo_json(name):
    local = os.path.join("..", "examples", "data", name)
    if os.path.exists(local):
        return json.load(open(local, encoding="utf-8"))
    url = f"https://raw.githubusercontent.com/Viny2030/qang/main/examples/data/{name}"
    return json.loads(urllib.request.urlopen(url).read().decode("utf-8"))


D = repo_json("lih_lambda_jw.json")
theta = repo_json("lih_lambda_filter_qg.json")["theta"]
NQ, PI_QUBITS = 5, (3, 4, 8, 9)  # qubit p = orbital p spin up, p + 5 spin down; orbitals 3, 4 = pi+, pi-
PARTNER = {0: 5, 1: 6, 2: 7, 3: 9, 4: 8}
KINDS = ("raw", "NS", "L", "NSL")
H = SparsePauliOp.from_list(list(D["terms"].items()))


def ansatz(th, basis=None):
    qc = QuantumCircuit(2 * NQ)
    qc.x(0)
    for k, t in enumerate(th):
        qc.append(XXPlusYYGate(2 * float(t)), [k, k + 1])
    for q, p in PARTNER.items():
        qc.cx(q, p)
    if basis is not None:
        for q, bb in enumerate(reversed(basis)):
            if bb == "X":
                qc.h(q)
            elif bb == "Y":
                qc.sdg(q); qc.h(q)
        qc.measure_all()
    return qc


def groups(H):
    diag, rest = [], []
    for lab, c in zip(H.paulis.to_labels(), H.coeffs):
        (diag if set(lab) <= set("IZ") else rest).append((lab, float(np.real(c))))
    pi_z = lambda lab: all(lab[2 * NQ - 1 - q] in "IZ" for q in PI_QUBITS)
    out = [diag]
    for part in ([t for t in rest if pi_z(t[0])], [t for t in rest if not pi_z(t[0])]):
        for g in SparsePauliOp.from_list(part).group_commuting(qubit_wise=True):
            out.append([(l, float(np.real(c))) for l, c in zip(g.paulis.to_labels(), g.coeffs)])
    return out


def basis_of(group):
    bs = ["Z"] * (2 * NQ)
    for lab, _ in group:
        for i, ch in enumerate(lab):
            if ch != "I":
                bs[i] = ch
    return "".join(bs)


def checks(basis):
    z = [basis[2 * NQ - 1 - q] == "Z" for q in range(2 * NQ)]
    return {"NS": all(z), "L": all(z[q] for q in PI_QUBITS)}


def keep(bit, kind, can):
    if kind in ("NS", "NSL") and can["NS"] and (sum(bit[:NQ]) != 1 or sum(bit[NQ:]) != 1):
        return False
    if kind in ("L", "NSL") and can["L"] and bit[3] + bit[8] - bit[4] - bit[9] != 0:
        return False
    return True


def estimate(counts, group, basis, kind):
    can = checks(basis)
    tot, acc = 0, np.zeros(len(group))
    for s, c in counts.items():
        bit = [int(s[2 * NQ - 1 - q]) for q in range(2 * NQ)]
        if not keep(bit, kind, can):
            continue
        tot += c
        for k, (lab, _) in enumerate(group):
            acc[k] += c * (1 - 2 * (sum(bit[q] for q in range(2 * NQ) if lab[2 * NQ - 1 - q] != "I") % 2))
    return float(sum(cf * v for (_, cf), v in zip(group, acc / max(tot, 1)))), tot


def pi_occupation(counts, kind):
    can = checks("Z" * (2 * NQ))
    tot = occ = 0
    for s, c in counts.items():
        bit = [int(s[2 * NQ - 1 - q]) for q in range(2 * NQ)]
        if keep(bit, kind, can):
            tot += c
            occ += c * sum(bit[q] for q in PI_QUBITS)
    return occ / max(tot, 1)


G = groups(H)
sv = Statevector(ansatz(theta))
exact = [float(np.real(sv.expectation_value(SparsePauliOp.from_list(g)))) for g in G]
print(f"{len(G)} measurement groups; N filter applies to {sum(checks(basis_of(g))['NS'] for g in G)}, "
      f"Lambda to {sum(checks(basis_of(g))['L'] for g in G)}; circuit energy {sum(exact):.5f} Ha "
      f"(CASCI {D['fci_active']:.5f})")

317 measurement groups; N filter applies to 1, Lambda to 29; circuit energy -7.87710 Ha (CASCI -7.88210)


In [8]:
b = fp.FakeTorino()
bases = [basis_of(g) for g in G]
ts = transpile([ansatz(theta, bs) for bs in bases], b, optimization_level=1, seed_transpiler=137)
try:
    from qiskit_ibm_runtime.executor_sampler import Sampler
except ImportError:
    from qiskit_ibm_runtime import SamplerV2 as Sampler
res = Sampler(mode=b).run(ts, shots=4000).result()
counts = [r.data.meas.get_counts() for r in res]
print("filter   energy error (mHa)   pi occupation (ideal %.4f)" % sum(
    p * sum(int(s[2 * NQ - 1 - q]) for q in PI_QUBITS) for s, p in sv.probabilities_dict().items()))
for kind, label in zip(KINDS, ("none", "N", "Lambda", "N + Lambda")):
    e = sum(estimate(c, g, bs, kind)[0] for c, g, bs in zip(counts, G, bases))
    print(f"{label:11s}  {1000 * abs(e - sum(exact)):8.1f}             {pi_occupation(counts[0], kind):.4f}")
print(f"total time {time.time() - T0:.0f} s")

filter   energy error (mHa)   pi occupation (ideal 0.0020)


none            173.3             0.1390


N                13.0             0.0027


Lambda          156.2             0.0755


N + Lambda       12.7             0.0014
total time 835 s


## Summary

* **Each quantum number has a qang read-out.** m_s is one qubit's qg_Z; m_l is the Hamming weight of a symmetric register, so the qg weight filter is the projection onto one m_l; l is read from qg_SWAP; n is the size of two registers. The qang angle of |l, m⟩ is arccos(m/l), not the vector-model angle. This is a translation, not new physics.
* **Λ on LiH**: a conserved quantity becomes a filter once the orbitals are chosen so that it is diagonal (complex π±). It adds little where the electron-number filter already applies, and it is the right check on the π occupation.

Details: `RESEARCH_NOTES.md` §153 and §155, `examples/quantum_numbers_qg.py`, `examples/lih_lambda_filter_qg.py`.